# Linear Regression Model — House Price Prediction

**Author:** Liz Choi  
**Fixes applied:**
- Changed `np.log1p` → `np.log` to be consistent with all other notebooks
- Added MAPE and MdAPE computed on the **actual ClosePrice scale** (after `np.exp()` back-transformation)
- Model v2 evaluation now includes full metric suite

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.model_selection import cross_val_score
import joblib

%matplotlib inline
sns.set(style="whitegrid")

## 2. Load Data

In [ ]:
train = pd.read_csv('train_cleaned.csv')
test  = pd.read_csv('test_cleaned.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")
train.head()

## 3. Log-Transform Target Variable

We apply `np.log()` (natural log) to `ClosePrice` to reduce right skew and improve model stability.  
Note: Using `np.log` (not `np.log1p`) to be consistent with all other notebooks in this project.

In [ ]:
# Use np.log (consistent with all other notebooks)
train['LogClosePrice'] = np.log(train['ClosePrice'])
test['LogClosePrice']  = np.log(test['ClosePrice'])

plt.figure(figsize=(10, 5))
sns.histplot(train['LogClosePrice'], kde=True)
plt.title('Distribution of Log-Transformed Close Price')
plt.xlabel('LogClosePrice')
plt.show()

## 4. Feature Selection (Numeric Features)

Start with core numeric features for the baseline linear regression model.

In [ ]:
features = ['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeAcres', 'YearBuilt']

X_train = train[features].fillna(0)
y_train = train['LogClosePrice']

X_test  = test[features].fillna(0)
y_test  = test['LogClosePrice']

## 5. Train Linear Regression Model (v1)

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

coeff_df = pd.DataFrame(model.coef_, features, columns=['Coefficient'])
print("Model Coefficients:")
print(coeff_df)
print(f"\nIntercept: {model.intercept_:.4f}")

## 6. Evaluate Model v1

MAPE and MdAPE are computed on the **actual ClosePrice scale** by back-transforming predictions with `np.exp()`.

In [ ]:
y_pred_log = model.predict(X_test)

# R² and RMSE on log scale
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))
r2_log   = r2_score(y_test, y_pred_log)

# Back-transform to actual price scale
y_test_actual = np.exp(y_test)
y_pred_actual = np.exp(y_pred_log)

# R² and RMSE on actual scale
rmse_actual = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
r2_actual   = r2_score(y_test_actual, y_pred_actual)

# MAPE and MdAPE on ACTUAL scale (corrected)
mape_actual  = mean_absolute_percentage_error(y_test_actual, y_pred_actual) * 100
mdape_actual = np.median(np.abs((y_test_actual - y_pred_actual) / y_test_actual)) * 100

print("=== Linear Regression v1 (Numeric Features Only) ===")
print(f"R² (log scale):    {r2_log:.4f}")
print(f"RMSE (log scale):  {rmse_log:.4f}")
print(f"R² (actual scale): {r2_actual:.4f}")
print(f"RMSE (actual $):   ${rmse_actual:,.0f}")
print(f"MAPE (actual):     {mape_actual:.2f}%")
print(f"MdAPE (actual):    {mdape_actual:.2f}%")

## 7. Actual vs Predicted Plot

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_log, alpha=0.3, s=10)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual LogClosePrice')
plt.ylabel('Predicted LogClosePrice')
plt.title('Actual vs Predicted Prices (Log Scale) — v1')
plt.tight_layout()
plt.show()

## 8. Residual Plot

In [ ]:
residuals = y_test - y_pred_log

plt.figure(figsize=(10, 6))
sns.residplot(x=y_pred_log, y=residuals, lowess=True, line_kws={'color': 'red', 'lw': 1})
plt.xlabel('Predicted LogClosePrice')
plt.ylabel('Residuals')
plt.title('Residual Plot (Checking for Heteroscedasticity)')
plt.axhline(y=0, color='black', linestyle='--')
plt.tight_layout()
plt.show()

## 9. Correlation Heatmap

In [ ]:
plt.figure(figsize=(12, 10))
numeric_cols = train.select_dtypes(include=[np.number]).columns
corr_matrix  = train[numeric_cols].corr()

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlation Heatmap of Numeric Features')
plt.tight_layout()
plt.show()

## 10. Model v2 — Add Categorical Features (One-Hot Encoding)

Extend the model by adding boolean categorical features.

In [ ]:
categorical_features = ['ViewYN', 'PoolPrivateYN', 'NewConstructionYN', 'FireplaceYN']

X_train_v2 = pd.get_dummies(train[features + categorical_features], drop_first=True)
X_test_v2  = pd.get_dummies(test[features + categorical_features],  drop_first=True)

# Align columns
X_test_v2 = X_test_v2.reindex(columns=X_train_v2.columns, fill_value=0)

model_v2 = LinearRegression()
model_v2.fit(X_train_v2, y_train)

y_pred_v2_log = model_v2.predict(X_test_v2)

## 11. Evaluate Model v2

Same correction applied: MAPE and MdAPE computed on **actual ClosePrice scale**.

In [ ]:
# R² on log scale
r2_v2_log  = r2_score(y_test, y_pred_v2_log)
rmse_v2_log = np.sqrt(mean_squared_error(y_test, y_pred_v2_log))

# Back-transform
y_pred_v2_actual = np.exp(y_pred_v2_log)

# R² and RMSE on actual scale
r2_v2_actual   = r2_score(y_test_actual, y_pred_v2_actual)
rmse_v2_actual = np.sqrt(mean_squared_error(y_test_actual, y_pred_v2_actual))

# MAPE and MdAPE on ACTUAL scale (corrected)
mape_v2_actual  = mean_absolute_percentage_error(y_test_actual, y_pred_v2_actual) * 100
mdape_v2_actual = np.median(np.abs((y_test_actual - y_pred_v2_actual) / y_test_actual)) * 100

print("=== Linear Regression v2 (Numeric + Categorical Features) ===")
print(f"R² (log scale):    {r2_v2_log:.4f}")
print(f"RMSE (log scale):  {rmse_v2_log:.4f}")
print(f"R² (actual scale): {r2_v2_actual:.4f}")
print(f"RMSE (actual $):   ${rmse_v2_actual:,.0f}")
print(f"MAPE (actual):     {mape_v2_actual:.2f}%")
print(f"MdAPE (actual):    {mdape_v2_actual:.2f}%")

## 12. Summary Table: v1 vs v2

In [ ]:
summary_df = pd.DataFrame({
    'Metric': [
        'R² (log scale)',
        'RMSE (log scale)',
        'R² (actual scale)',
        'RMSE (actual $)',
        'MAPE (%) — actual scale',
        'MdAPE (%) — actual scale'
    ],
    'v1 (Numeric only)': [
        round(r2_log,       4),
        round(rmse_log,     4),
        round(r2_actual,    4),
        round(rmse_actual,  0),
        round(mape_actual,  2),
        round(mdape_actual, 2)
    ],
    'v2 (Numeric + Categorical)': [
        round(r2_v2_log,       4),
        round(rmse_v2_log,     4),
        round(r2_v2_actual,    4),
        round(rmse_v2_actual,  0),
        round(mape_v2_actual,  2),
        round(mdape_v2_actual, 2)
    ]
})

summary_df

## 13. Cross-Validation on v1 (5-Fold)

In [ ]:
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')

print(f"Cross-Validation R² Scores: {cv_scores.round(4)}")
print(f"Mean CV R²: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

## 14. Prediction Sample (Actual Dollar Scale)

In [ ]:
comparison_df = pd.DataFrame({
    'Actual ($)':    y_test_actual.values,
    'Predicted ($)': y_pred_actual.values
})
comparison_df['Abs_Error ($)']  = (comparison_df['Actual ($)'] - comparison_df['Predicted ($)']).abs()
comparison_df['Pct_Error (%)']  = (comparison_df['Abs_Error ($)'] / comparison_df['Actual ($)'] * 100).round(2)

print("Sample predictions (v1):")
comparison_df.head(10)

## 15. Save Model v2

In [ ]:
joblib.dump(model_v2, 'linear_regression_house_model.pkl')
print("Model saved as linear_regression_house_model.pkl")

# To load later:
# loaded_model = joblib.load('linear_regression_house_model.pkl')